In [ ]:
import json
import os
import glob
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Make the project root importable so `from src.bots import ...` works
sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
from src.artifacts_store import (
    apply_bot_cache,
    load_classifier,
    try_load_game_cache,
)

N_PREVIEW = 5

re_human_cache = try_load_game_cache("re", "human")
re_bot_cache = try_load_game_cache("re", "bot")
if re_human_cache is None or re_bot_cache is None:
    raise FileNotFoundError("re cache")
re_games_df = re_human_cache["features"]
apply_bot_cache("re", re_bot_cache, globals(), n_preview=N_PREVIEW)

lol_human_cache = try_load_game_cache("lol", "human")
lol_bot_cache = try_load_game_cache("lol", "bot")
if lol_human_cache is None or lol_bot_cache is None:
    raise FileNotFoundError("lol cache")
lol_games_df = lol_human_cache["features"]
lol_mouse_by_game = dict(zip(lol_human_cache["session_ids"], lol_human_cache["traces"]))
apply_bot_cache("lol", lol_bot_cache, globals(), n_preview=N_PREVIEW)

csgo_human_cache = try_load_game_cache("csgo", "human")
csgo_bot_cache = try_load_game_cache("csgo", "bot")
if csgo_human_cache is None or csgo_bot_cache is None:
    raise FileNotFoundError("csgo cache")
csgo_games_df = csgo_human_cache["features"]
csgo_mouse_win = {}
for sid, trace in zip(csgo_human_cache["session_ids"], csgo_human_cache["traces"]):
    session, participant = sid.split("_", 1)
    csgo_mouse_win[(session, participant)] = trace
apply_bot_cache("csgo", csgo_bot_cache, globals(), n_preview=N_PREVIEW)

for bot_type in ("stitch", "smooth", "bezier", "vae"):
    globals()[f"re_model_{bot_type}"] = load_classifier("re", "raw", bot_type)
    globals()[f"m_si_{bot_type}"] = load_classifier("re", "si_min", bot_type)
    globals()[f"m_si_ext_{bot_type}"] = load_classifier("re", "si_ext", bot_type)
    globals()[f"csgo_x_model_{bot_type}"] = load_classifier("csgo", "raw", bot_type)
    globals()[f"m_csgo_si_{bot_type}"] = load_classifier("csgo", "si_min", bot_type)
    globals()[f"m_csgo_si_ext_{bot_type}"] = load_classifier("csgo", "si_ext", bot_type)


## Feature scale comparison (Red Eclipse vs LoL)

In [ ]:
from src.features import cross_game_feature_cols

cols = cross_game_feature_cols

print("Feature medians (Red Eclipse human vs LoL human vs LoL bots):")
compare = pd.DataFrame({
    "Red Eclipse human": re_games_df[cols].median(),
    "LoL_human": lol_games_df[cols].median(),
    "LoL Stitch": lol_bots_stitch_df[cols].median(),
    "LoL Scripted": lol_bots_smooth_df[cols].median(),
    "LoL Bézier": lol_bots_bezier_df[cols].median(),
    "LoL VAE": lol_bots_vae_df[cols].median(),
}).round(3)
print(compare)


## Red Eclipse → LoL diagnose (raw features)

In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import cross_game_feature_cols

print("=== Raw features: true zero-shot Red Eclipse -> LoL ===")
print("(threshold from LoL humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", re_model_stitch, lol_games_df, lol_bots_stitch_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "smooth", re_model_smooth, lol_games_df, lol_bots_smooth_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "bezier", re_model_bezier, lol_games_df, lol_bots_bezier_df,
    cross_game_feature_cols, title_suffix="raw",
)

print()
_ = diagnose_cross_game(
    "vae", re_model_vae, lol_games_df, lol_bots_vae_df,
    cross_game_feature_cols, title_suffix="raw",
)


## Red Eclipse → LoL diagnose (SI-min(4))

In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_COLS

lol_si_human = to_scale_invariant(lol_games_df)
lol_si_stitch = to_scale_invariant(lol_bots_stitch_df)
lol_si_smooth = to_scale_invariant(lol_bots_smooth_df)
lol_si_bezier = to_scale_invariant(lol_bots_bezier_df)
lol_si_vae = to_scale_invariant(lol_bots_vae_df)

print("=== SI-min(4): true zero-shot Red Eclipse -> LoL ===")
print("(threshold from LoL humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", m_si_stitch, lol_si_human, lol_si_stitch,
    SCALE_INVARIANT_COLS, title_suffix="SI-min(4)",
)
print()
_ = diagnose_cross_game(
    "smooth", m_si_smooth, lol_si_human, lol_si_smooth,
    SCALE_INVARIANT_COLS, title_suffix="SI-min(4)",
)
print()
_ = diagnose_cross_game(
    "bezier", m_si_bezier, lol_si_human, lol_si_bezier,
    SCALE_INVARIANT_COLS, title_suffix="SI-min(4)",
)

print()
_ = diagnose_cross_game(
    "vae", m_si_vae, lol_si_human, lol_si_vae,
    SCALE_INVARIANT_COLS, title_suffix="SI-min(4)",
)


## Red Eclipse → LoL diagnose (SI-ext(10))


In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import SCALE_INVARIANT_EXT_COLS

print("=== SI-ext(10): true zero-shot Red Eclipse -> LoL ===")
print("(threshold from LoL humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", m_si_ext_stitch, lol_si_human, lol_si_stitch,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-ext(10)",
)
print()
_ = diagnose_cross_game(
    "smooth", m_si_ext_smooth, lol_si_human, lol_si_smooth,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-ext(10)",
)
print()
_ = diagnose_cross_game(
    "bezier", m_si_ext_bezier, lol_si_human, lol_si_bezier,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-ext(10)",
)
print()
_ = diagnose_cross_game(
    "vae", m_si_ext_vae, lol_si_human, lol_si_vae,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-ext(10)",
)


## SI-ext(10) ablation: drop `xy_corr` / `vh_ratio` (Red Eclipse → LoL)

In [ ]:
import pandas as pd
from src.evaluation import train_bot_detector, diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_EXT_COLS
from src.config import RNG_SEED

EXT_FULL = list(SCALE_INVARIANT_EXT_COLS)
EXT_NO_XY = [c for c in EXT_FULL if c != "xy_corr"]
EXT_NO_XY_VH = [c for c in EXT_NO_XY if c != "vh_ratio"]

re_si_human = to_scale_invariant(re_games_df)
re_bots = {
    "stitch": to_scale_invariant(re_stitch_df),
    "smooth": to_scale_invariant(re_smooth_df),
    "bezier": to_scale_invariant(re_bezier_df),
    "vae": to_scale_invariant(re_vae_df),
}
lol_si_human = to_scale_invariant(lol_games_df)
lol_bots = {
    "stitch": to_scale_invariant(lol_bots_stitch_df),
    "smooth": to_scale_invariant(lol_bots_smooth_df),
    "bezier": to_scale_invariant(lol_bots_bezier_df),
    "vae": to_scale_invariant(lol_bots_vae_df),
}

ablations = {
    "EXT-10": EXT_FULL,
    "EXT-xy": EXT_NO_XY,
    "EXT-xy-vh": EXT_NO_XY_VH,
}

rows = []
for abl_name, cols in ablations.items():
    print(f"\n{'=' * 60}")
    print(f"=== Ablation {abl_name} ({len(cols)} feats): {cols} ===")
    for bot in ["stitch", "smooth", "bezier", "vae"]:
        model, _ = train_bot_detector(
            re_si_human, re_bots[bot], cols,
            random_state=RNG_SEED, name=f"{abl_name} {bot}",
            show_feature_importance=False,
        )
        d = diagnose_cross_game(
            bot, model, lol_si_human, lol_bots[bot], cols,
            title_suffix=f"Red Eclipse→LoL {abl_name}",
            plot=False, plot_proba=False,
        )
        rows.append({
            "ablation": abl_name,
            "n_feats": len(cols),
            "bot": bot,
            "auc": d["auc"],
            "cal_detect": d["detect_cal"],
            "cal_fp": d["fp_cal"],
            "thr": d["thr"],
            "detect_05": d["detect_05"],
            "fp_05": d["fp_05"],
        })
        print()

summary = pd.DataFrame(rows)
print("\n=== Red Eclipse→LoL SI-ext(10) ablation summary ===")
print(
    summary.assign(bot=summary["bot"].replace(
        {"stitch": "Stitch", "smooth": "Scripted", "bezier": "Bézier", "vae": "VAE"}
    )).pivot_table(
        index="bot", columns="ablation",
        values=["auc", "cal_detect"],
    ).round(3).to_string()
)
print("\n(VAE focus: does cal_detect stay high after dropping vh_ratio?)")
print(summary[summary["bot"] == "vae"][
    ["ablation", "n_feats", "auc", "cal_detect", "cal_fp", "thr"]
].to_string(index=False))


## Red Eclipse→LoL Bézier inversion diagnostic


In [ ]:
import pandas as pd
from src.features import cross_game_feature_cols, to_scale_invariant, SCALE_INVARIANT_COLS

raw_cols = cross_game_feature_cols

# --- (1) Feature medians: RE human / RE bezier / LoL human / LoL bezier ---
print("=== Raw 7-feature medians ===")
raw_med = pd.DataFrame({
    "Red Eclipse human": re_games_df[raw_cols].median(),
    "Red Eclipse Bézier": re_bezier_df[raw_cols].median(),
    "LoL_human": lol_games_df[raw_cols].median(),
    "LoL Bézier": lol_bots_bezier_df[raw_cols].median(),
}).round(4)
# signed gap human - bot (same game); flip if Red Eclipse and LoL gaps have opposite sign
raw_med["gap_Red Eclipse"] = (raw_med["Red Eclipse human"] - raw_med["Red Eclipse Bézier"]).round(4)
raw_med["gap_LoL"] = (raw_med["LoL_human"] - raw_med["LoL Bézier"]).round(4)
raw_med["sign_flip"] = (raw_med["gap_Red Eclipse"] * raw_med["gap_LoL"]) < 0
print(raw_med)
print()

re_sf = to_scale_invariant(re_games_df)
re_bz_sf = to_scale_invariant(re_bezier_df)
lol_sf = to_scale_invariant(lol_games_df)
lol_bz_sf = to_scale_invariant(lol_bots_bezier_df)

print("=== SI-min(4) medians ===")
sf_med = pd.DataFrame({
    "Red Eclipse human": re_sf[SCALE_INVARIANT_COLS].median(),
    "Red Eclipse Bézier": re_bz_sf[SCALE_INVARIANT_COLS].median(),
    "LoL_human": lol_sf[SCALE_INVARIANT_COLS].median(),
    "LoL Bézier": lol_bz_sf[SCALE_INVARIANT_COLS].median(),
}).round(4)
sf_med["gap_Red Eclipse"] = (sf_med["Red Eclipse human"] - sf_med["Red Eclipse Bézier"]).round(4)
sf_med["gap_LoL"] = (sf_med["LoL_human"] - sf_med["LoL Bézier"]).round(4)
sf_med["sign_flip"] = (sf_med["gap_Red Eclipse"] * sf_med["gap_LoL"]) < 0
print(sf_med)
print()

# --- (2) Feature importance of the RE-trained bezier detectors ---
print("=== Red Eclipse Bézier model importance (raw 7-feat, used in Red Eclipse→LoL raw) ===")
imp_raw = pd.Series(
    re_model_bezier.feature_importances_, index=raw_cols
).sort_values(ascending=False)
print(imp_raw.round(4))
print()

print("=== Red Eclipse Bézier model importance (SI-min(4), used in Red Eclipse→LoL SF) ===")
imp_sf = pd.Series(
    m_si_bezier.feature_importances_, index=SCALE_INVARIANT_COLS
).sort_values(ascending=False)
print(imp_sf.round(4))
print()

# --- Cross-read: high importance ∩ sign flip ---
print("=== Suspects: importance rank + sign_flip ===")
print("Raw:")
for feat, imp in imp_raw.items():
    flip = bool(raw_med.loc[feat, "sign_flip"])
    print(f"  {feat:16s}  imp={imp:.3f}  flip={flip}  "
          f"gap_Red Eclipse={raw_med.loc[feat, 'gap_Red Eclipse']:+.4f}  gap_LoL={raw_med.loc[feat, 'gap_LoL']:+.4f}")
print("SI-min(4):")
for feat, imp in imp_sf.items():
    flip = bool(sf_med.loc[feat, "sign_flip"])
    print(f"  {feat:16s}  imp={imp:.3f}  flip={flip}  "
          f"gap_Red Eclipse={sf_med.loc[feat, 'gap_Red Eclipse']:+.4f}  gap_LoL={sf_med.loc[feat, 'gap_LoL']:+.4f}")



## Red Eclipse→LoL Bézier diagnostic (SI-ext(10) medians)


In [ ]:
import pandas as pd
from src.features import to_scale_invariant, SCALE_INVARIANT_EXT_COLS

re_sf = to_scale_invariant(re_games_df)
re_bz_sf = to_scale_invariant(re_bezier_df)
lol_sf = to_scale_invariant(lol_games_df)
lol_bz_sf = to_scale_invariant(lol_bots_bezier_df)

print("=== SI-ext(10) 10-feature medians ===")
sf_med = pd.DataFrame({
    "Red Eclipse human": re_sf[SCALE_INVARIANT_EXT_COLS].median(),
    "Red Eclipse Bézier": re_bz_sf[SCALE_INVARIANT_EXT_COLS].median(),
    "LoL_human": lol_sf[SCALE_INVARIANT_EXT_COLS].median(),
    "LoL Bézier": lol_bz_sf[SCALE_INVARIANT_EXT_COLS].median(),
}).round(4)
sf_med["gap_Red Eclipse"] = (sf_med["Red Eclipse human"] - sf_med["Red Eclipse Bézier"]).round(4)
sf_med["gap_LoL"] = (sf_med["LoL_human"] - sf_med["LoL Bézier"]).round(4)
sf_med["sign_flip"] = (sf_med["gap_Red Eclipse"] * sf_med["gap_LoL"]) < 0
print(sf_med)
print()

print("=== Red Eclipse Bézier model importance (SI-ext(10), used in Red Eclipse→LoL EXT) ===")
imp_sf = pd.Series(
    m_si_ext_bezier.feature_importances_, index=SCALE_INVARIANT_EXT_COLS
).sort_values(ascending=False)
print(imp_sf.round(4))
print()

print("=== Suspects: SI-ext(10) importance + sign_flip ===")
for feat, imp in imp_sf.items():
    flip = bool(sf_med.loc[feat, "sign_flip"])
    print(f"  {feat:16s}  imp={imp:.3f}  flip={flip}  "
          f"gap_Red Eclipse={sf_med.loc[feat, 'gap_Red Eclipse']:+.4f}  gap_LoL={sf_med.loc[feat, 'gap_LoL']:+.4f}")


## Feature importance (SI-min(4) models)



In [ ]:
import pandas as pd
from src.features import SCALE_INVARIANT_COLS, SCALE_INVARIANT_EXT_COLS

imp = pd.Series(m_si_stitch.feature_importances_, index=SCALE_INVARIANT_COLS).sort_values(ascending=False)
print("=== Feature importance (SI-min(4) Stitch model) ===")
print(imp)
print()
imp2 = pd.Series(m_si_smooth.feature_importances_, index=SCALE_INVARIANT_COLS).sort_values(ascending=False)
print("=== Feature importance (SI-min(4) Scripted model) ===")
print(imp2)
print()
imp3 = pd.Series(m_si_bezier.feature_importances_, index=SCALE_INVARIANT_COLS).sort_values(ascending=False)
print("=== Feature importance (SI-min(4) Bézier model) ===")
print(imp3)
print()
imp4 = pd.Series(m_si_vae.feature_importances_, index=SCALE_INVARIANT_COLS).sort_values(ascending=False)
print("=== Feature importance (SI-min(4) VAE model) ===")
print(imp4)

imp_ext = pd.Series(m_si_ext_stitch.feature_importances_, index=SCALE_INVARIANT_EXT_COLS).sort_values(ascending=False)
print("=== Feature importance (SI-ext(10) Stitch) ===")
print(imp_ext)
print()
imp_ext2 = pd.Series(m_si_ext_smooth.feature_importances_, index=SCALE_INVARIANT_EXT_COLS).sort_values(ascending=False)
print("=== Feature importance (SI-ext(10) Scripted) ===")
print(imp_ext2)
print()
imp_ext3 = pd.Series(m_si_ext_bezier.feature_importances_, index=SCALE_INVARIANT_EXT_COLS).sort_values(ascending=False)
print("=== Feature importance (SI-ext(10) Bézier) ===")
print(imp_ext3)
print()
imp_ext4 = pd.Series(m_si_ext_vae.feature_importances_, index=SCALE_INVARIANT_EXT_COLS).sort_values(ascending=False)
print("=== Feature importance (SI-ext(10) VAE) ===")
print(imp_ext4)

## Zero-shot diagnose (raw features, Red Eclipse → CSGO)

In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import cross_game_feature_cols

print("=== Raw features: true zero-shot Red Eclipse -> CSGO ===")
print("(threshold from CSGO humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", re_model_stitch, csgo_games_df, csgo_bots_stitch_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "smooth", re_model_smooth, csgo_games_df, csgo_bots_smooth_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "bezier", re_model_bezier, csgo_games_df, csgo_bots_bezier_df,
    cross_game_feature_cols, title_suffix="raw",
)

print()
_ = diagnose_cross_game(
    "vae", re_model_vae, csgo_games_df, csgo_bots_vae_df,
    cross_game_feature_cols, title_suffix="raw",
)


## Zero-shot diagnose (SI-min(4), Red Eclipse → CSGO)


In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_COLS

csgo_si_human = to_scale_invariant(csgo_games_df)
csgo_si_stitch = to_scale_invariant(csgo_bots_stitch_df)
csgo_si_smooth = to_scale_invariant(csgo_bots_smooth_df)
csgo_si_bezier = to_scale_invariant(csgo_bots_bezier_df)
csgo_si_vae = to_scale_invariant(csgo_bots_vae_df)

print("=== SI-min(4): true zero-shot Red Eclipse -> CSGO ===")
print("(threshold from CSGO humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", m_si_stitch, csgo_si_human, csgo_si_stitch,
    SCALE_INVARIANT_COLS, title_suffix="SI-min(4)",
)
print()
_ = diagnose_cross_game(
    "smooth", m_si_smooth, csgo_si_human, csgo_si_smooth,
    SCALE_INVARIANT_COLS, title_suffix="SI-min(4)",
)
print()
_ = diagnose_cross_game(
    "bezier", m_si_bezier, csgo_si_human, csgo_si_bezier,
    SCALE_INVARIANT_COLS, title_suffix="SI-min(4)",
)

print()
_ = diagnose_cross_game(
    "vae", m_si_vae, csgo_si_human, csgo_si_vae,
    SCALE_INVARIANT_COLS, title_suffix="SI-min(4)",
)


## Zero-shot diagnose (SI-ext(10), Red Eclipse → CSGO)


In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_EXT_COLS

csgo_si_human = to_scale_invariant(csgo_games_df)
csgo_si_stitch = to_scale_invariant(csgo_bots_stitch_df)
csgo_si_smooth = to_scale_invariant(csgo_bots_smooth_df)
csgo_si_bezier = to_scale_invariant(csgo_bots_bezier_df)
csgo_si_vae = to_scale_invariant(csgo_bots_vae_df)

print("=== SI-ext(10): true zero-shot Red Eclipse -> CSGO ===")
print("(threshold from CSGO humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", m_si_ext_stitch, csgo_si_human, csgo_si_stitch,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-ext(10)",
)
print()
_ = diagnose_cross_game(
    "smooth", m_si_ext_smooth, csgo_si_human, csgo_si_smooth,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-ext(10)",
)
print()
_ = diagnose_cross_game(
    "bezier", m_si_ext_bezier, csgo_si_human, csgo_si_bezier,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-ext(10)",
)
print()
_ = diagnose_cross_game(
    "vae", m_si_ext_vae, csgo_si_human, csgo_si_vae,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-ext(10)",
)


## SI-ext(10) ablation: drop `xy_corr` / `vh_ratio` (Red Eclipse → CSGO)


In [ ]:
import pandas as pd
from src.evaluation import train_bot_detector, diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_EXT_COLS
from src.config import RNG_SEED

EXT_FULL = list(SCALE_INVARIANT_EXT_COLS)
EXT_NO_XY = [c for c in EXT_FULL if c != "xy_corr"]
EXT_NO_XY_VH = [c for c in EXT_NO_XY if c != "vh_ratio"]

re_si_human = to_scale_invariant(re_games_df)
re_bots = {
    "stitch": to_scale_invariant(re_stitch_df),
    "smooth": to_scale_invariant(re_smooth_df),
    "bezier": to_scale_invariant(re_bezier_df),
    "vae": to_scale_invariant(re_vae_df),
}
csgo_si_human = to_scale_invariant(csgo_games_df)
csgo_bots = {
    "stitch": to_scale_invariant(csgo_bots_stitch_df),
    "smooth": to_scale_invariant(csgo_bots_smooth_df),
    "bezier": to_scale_invariant(csgo_bots_bezier_df),
    "vae": to_scale_invariant(csgo_bots_vae_df),
}

ablations = {
    "EXT-10": EXT_FULL,
    "EXT-xy": EXT_NO_XY,
    "EXT-xy-vh": EXT_NO_XY_VH,
}

rows = []
for abl_name, cols in ablations.items():
    print(f"\n{'=' * 60}")
    print(f"=== Ablation {abl_name} ({len(cols)} feats) Red Eclipse→CSGO ===")
    for bot in ["stitch", "smooth", "bezier", "vae"]:
        model, _ = train_bot_detector(
            re_si_human, re_bots[bot], cols,
            random_state=RNG_SEED, name=f"{abl_name} {bot}",
            show_feature_importance=False,
        )
        d = diagnose_cross_game(
            bot, model, csgo_si_human, csgo_bots[bot], cols,
            title_suffix=f"Red Eclipse→CSGO {abl_name}",
            plot=False, plot_proba=False,
        )
        rows.append({
            "ablation": abl_name, "n_feats": len(cols), "bot": bot,
            "auc": d["auc"], "cal_detect": d["detect_cal"],
            "cal_fp": d["fp_cal"], "thr": d["thr"],
        })
        print()

summary = pd.DataFrame(rows)
print("\n=== Red Eclipse→CSGO SI-ext(10) ablation summary ===")
print(
    summary.assign(bot=summary["bot"].replace(
        {"stitch": "Stitch", "smooth": "Scripted", "bezier": "Bézier", "vae": "VAE"}
    )).pivot_table(
        index="bot", columns="ablation", values=["auc", "cal_detect"],
    ).round(3).to_string()
)
print("\nVAE focus:")
print(summary[summary["bot"] == "vae"][
    ["ablation", "n_feats", "auc", "cal_detect", "cal_fp", "thr"]
].to_string(index=False))


## Feature scale comparison (Red Eclipse vs CSGO)

In [ ]:
from src.features import cross_game_feature_cols

cols = cross_game_feature_cols

print("Feature medians (Red Eclipse human vs CSGO human vs CSGO bots):")
compare_re_csgo = pd.DataFrame({
    "Red Eclipse human": re_games_df[cols].median(),
    "CSGO_human": csgo_games_df[cols].median(),
    "CSGO Stitch": csgo_bots_stitch_df[cols].median(),
    "CSGO Scripted": csgo_bots_smooth_df[cols].median(),
    "CSGO Bézier": csgo_bots_bezier_df[cols].median(),
    "CSGO VAE": csgo_bots_vae_df[cols].median(),
}).round(3)
print(compare_re_csgo)



## Zero-shot diagnose (raw features, CSGO → Red Eclipse)

In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import cross_game_feature_cols

print("=== Raw features: true zero-shot CSGO -> Red Eclipse ===")
print("(threshold from Red Eclipse humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", csgo_x_model_stitch, re_games_df, re_stitch_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "smooth", csgo_x_model_smooth, re_games_df, re_smooth_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "bezier", csgo_x_model_bezier, re_games_df, re_bezier_df,
    cross_game_feature_cols, title_suffix="raw",
)

print()
_ = diagnose_cross_game(
    "vae", csgo_x_model_vae, re_games_df, re_vae_df,
    cross_game_feature_cols, title_suffix="raw",
)


## Zero-shot diagnose (SI-min(4), CSGO → Red Eclipse)



In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_COLS

re_si_human = to_scale_invariant(re_games_df)
re_si_stitch = to_scale_invariant(re_stitch_df)
re_si_smooth = to_scale_invariant(re_smooth_df)
re_si_bezier = to_scale_invariant(re_bezier_df)
re_si_vae = to_scale_invariant(re_vae_df)

print("=== SI-min(4): true zero-shot CSGO -> Red Eclipse ===")
print("(threshold from Red Eclipse humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", m_csgo_si_stitch, re_si_human, re_si_stitch,
    SCALE_INVARIANT_COLS, title_suffix="SI-min(4)",
)
print()
_ = diagnose_cross_game(
    "smooth", m_csgo_si_smooth, re_si_human, re_si_smooth,
    SCALE_INVARIANT_COLS, title_suffix="SI-min(4)",
)
print()
_ = diagnose_cross_game(
    "bezier", m_csgo_si_bezier, re_si_human, re_si_bezier,
    SCALE_INVARIANT_COLS, title_suffix="SI-min(4)",
)

print()
_ = diagnose_cross_game(
    "vae", m_csgo_si_vae, re_si_human, re_si_vae,
    SCALE_INVARIANT_COLS, title_suffix="SI-min(4)",
)


## Zero-shot diagnose (SI-ext(10), CSGO → Red Eclipse)


In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_EXT_COLS

re_si_human = to_scale_invariant(re_games_df)
re_si_stitch = to_scale_invariant(re_stitch_df)
re_si_smooth = to_scale_invariant(re_smooth_df)
re_si_bezier = to_scale_invariant(re_bezier_df)
re_si_vae = to_scale_invariant(re_vae_df)

print("=== SI-ext(10): true zero-shot CSGO -> Red Eclipse ===")
print("(threshold from Red Eclipse humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", m_csgo_si_ext_stitch, re_si_human, re_si_stitch,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-ext(10)",
)
print()
_ = diagnose_cross_game(
    "smooth", m_csgo_si_ext_smooth, re_si_human, re_si_smooth,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-ext(10)",
)
print()
_ = diagnose_cross_game(
    "bezier", m_csgo_si_ext_bezier, re_si_human, re_si_bezier,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-ext(10)",
)
print()
_ = diagnose_cross_game(
    "vae", m_csgo_si_ext_vae, re_si_human, re_si_vae,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-ext(10)",
)


## SI-ext(10) ablation: drop `xy_corr` / `vh_ratio` (CSGO → Red Eclipse)


In [ ]:
import pandas as pd
from src.evaluation import train_bot_detector, diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_EXT_COLS
from src.config import RNG_SEED

EXT_FULL = list(SCALE_INVARIANT_EXT_COLS)
EXT_NO_XY = [c for c in EXT_FULL if c != "xy_corr"]
EXT_NO_XY_VH = [c for c in EXT_NO_XY if c != "vh_ratio"]

csgo_si_human_tr = to_scale_invariant(csgo_games_df)
csgo_bots_tr = {
    "stitch": to_scale_invariant(csgo_bots_stitch_df),
    "smooth": to_scale_invariant(csgo_bots_smooth_df),
    "bezier": to_scale_invariant(csgo_bots_bezier_df),
    "vae": to_scale_invariant(csgo_bots_vae_df),
}
re_si_human = to_scale_invariant(re_games_df)
re_bots = {
    "stitch": to_scale_invariant(re_stitch_df),
    "smooth": to_scale_invariant(re_smooth_df),
    "bezier": to_scale_invariant(re_bezier_df),
    "vae": to_scale_invariant(re_vae_df),
}

ablations = {
    "EXT-10": EXT_FULL,
    "EXT-xy": EXT_NO_XY,
    "EXT-xy-vh": EXT_NO_XY_VH,
}

rows = []
for abl_name, cols in ablations.items():
    print(f"\n{'=' * 60}")
    print(f"=== Ablation {abl_name} ({len(cols)} feats) CSGO→Red Eclipse ===")
    for bot in ["stitch", "smooth", "bezier", "vae"]:
        model, _ = train_bot_detector(
            csgo_si_human_tr, csgo_bots_tr[bot], cols,
            random_state=RNG_SEED, name=f"CSGO {abl_name} {bot}",
            show_feature_importance=False,
        )
        d = diagnose_cross_game(
            bot, model, re_si_human, re_bots[bot], cols,
            title_suffix=f"CSGO→Red Eclipse {abl_name}",
            plot=False, plot_proba=False,
        )
        rows.append({
            "ablation": abl_name, "n_feats": len(cols), "bot": bot,
            "auc": d["auc"], "cal_detect": d["detect_cal"],
            "cal_fp": d["fp_cal"], "thr": d["thr"],
        })
        print()

summary = pd.DataFrame(rows)
print("\n=== CSGO→Red Eclipse SI-ext(10) ablation summary ===")
print(
    summary.assign(bot=summary["bot"].replace(
        {"stitch": "Stitch", "smooth": "Scripted", "bezier": "Bézier", "vae": "VAE"}
    )).pivot_table(
        index="bot", columns="ablation", values=["auc", "cal_detect"],
    ).round(3).to_string()
)
print("\nVAE focus:")
print(summary[summary["bot"] == "vae"][
    ["ablation", "n_feats", "auc", "cal_detect", "cal_fp", "thr"]
].to_string(index=False))


## Feature scale comparison (CSGO vs Red Eclipse)

In [ ]:
from src.features import cross_game_feature_cols

cols = cross_game_feature_cols

print("Feature medians (CSGO human vs Red Eclipse human vs Red Eclipse bots):")
compare_csgo_re = pd.DataFrame({
    "CSGO_human": csgo_games_df[cols].median(),
    "Red Eclipse human": re_games_df[cols].median(),
    "Red Eclipse Stitch": re_stitch_df[cols].median(),
    "Red Eclipse Scripted": re_smooth_df[cols].median(),
    "Red Eclipse Bézier": re_bezier_df[cols].median(),
    "Red Eclipse VAE": re_vae_df[cols].median(),
}).round(3)
print(compare_csgo_re)

